# DE2 — Lab 1: Structured Streaming Pipeline (10%)
> Author : Badr TAJINI - Data Engineering II (Data-Intensive Workloads) - ESIEE 2025-2026

**Track:** A *(Write your track: A/B/C/D)*

**Goal:** Build a Structured Streaming pipeline with windowed aggregation, watermarks, and a Parquet sink. Monitor via `query.lastProgress` and the Streaming UI. Deliver a before/after optimization report.

In [1]:
import sys
import os

print("Exécutable Python :", sys.executable)
print("Dossier de travail :", os.getcwd())

Exécutable Python : /usr/bin/python3
Dossier de travail : /mnt/c/Users/flori/OneDrive/Documents/Nouvelle_Sauvegarde/ESIEE/E4FD/Data_engineering_TAJINI/DE2


In [10]:
import os
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_DRIVER_BIND_ADDRESS"] = "127.0.0.1"

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *
import time, pathlib, json

spark = SparkSession.builder \
    .appName("DE2-Lab1-Streaming") \
    .master("local[*]") \
    .getOrCreate()

print("Spark version:", spark.version)
print("Spark UI:", spark.sparkContext.uiWebUrl)

Spark version: 4.1.1
Spark UI: http://localhost:4040


## 1. Define Schema & Prepare Landing Directory

Define an explicit `StructType` schema for your track data. Create a landing directory where files will be dropped to simulate a streaming source.

In [5]:
%pip install pandas --break-system-packages

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 7.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 6.7 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [11]:
# TODO: Define your schema (adapt columns to your track)
# schema = StructType([
#     StructField("event_time", TimestampType(), True),
#     StructField("key_col",    StringType(),    True),
#     StructField("metric",     DoubleType(),    True),
# ])
schema = StructType([
    StructField("date",         TimestampType(), True),
    StructField("team_1",       StringType(),    True),
    StructField("team_2",       StringType(),    True),
    StructField("_map",         StringType(),    True),
    StructField("result_1",     IntegerType(),   True),
    StructField("result_2",     IntegerType(),   True),
    StructField("map_winner",   IntegerType(),   True), # Souvent 1 ou 2
    StructField("starting_ct",  IntegerType(),   True), # Souvent 1 ou 2
    StructField("ct_1",         IntegerType(),   True),
    StructField("t_2",          IntegerType(),   True),
    StructField("t_1",          IntegerType(),   True),
    StructField("ct_2",         IntegerType(),   True),
    StructField("event_id",     IntegerType(),   True),
    StructField("match_id",     IntegerType(),   True),
    StructField("rank_1",       IntegerType(),   True),
    StructField("rank_2",       IntegerType(),   True),
    StructField("map_wins_1",   IntegerType(),   True),
    StructField("map_wins_2",   IntegerType(),   True),
    StructField("match_winner", IntegerType(),   True)  # Souvent 1 ou 2
])

# TODO: Create landing directory
# landing_dir = "data/landing/lab1/"
# pathlib.Path(landing_dir).mkdir(parents=True, exist_ok=True)
landing_dir = "data/landing/csgo_matches/"
pathlib.Path(landing_dir).mkdir(parents=True, exist_ok=True)
print(f"Le dossier {landing_dir} a été créé/vérifié avec succès.")

# Optional: copy a few small CSV files into landing_dir to simulate streaming
# Example: split your dataset into 5 CSV files and drop them one by one
def simulate_csv_streaming(source_csv_path, landing_directory, num_chunks=5):
    """
    Lit le CSV source, le divise en plusieurs petits fichiers CSV 
    et les sauvegarde dans le dossier landing_dir.
    """
    try:
        # Lire le dataset complet
        df = pd.read_csv(source_csv_path)
        
        # Calculer la taille de chaque morceau
        chunk_size = len(df) // num_chunks
        
        for i in range(num_chunks):
            start_idx = i * chunk_size
            # Pour le dernier chunk, on prend tout le reste
            end_idx = (i + 1) * chunk_size if i < num_chunks - 1 else len(df)
            
            # Découper le dataframe
            chunk = df.iloc[start_idx:end_idx]
            
            # Sauvegarder dans le landing_dir
            output_file = f"{landing_directory}/match_data_part_{i+1}.csv"
            chunk.to_csv(output_file, index=False)
            print(f"Fichier créé : {output_file} ({len(chunk)} lignes)")
            
    except FileNotFoundError:
        print(f"Erreur : Le fichier source '{source_csv_path}' est introuvable.")

Le dossier data/landing/csgo_matches/ a été créé/vérifié avec succès.


## 2. Create Streaming Source

Use `spark.readStream` to create a streaming DataFrame from the landing directory.

In [12]:
# TODO: Create streaming source
# df_stream = (spark.readStream
#     .schema(schema)
#     .option("header", "true")
#     .option("maxFilesPerTrigger", 1)  # process one file per micro-batch
#     .csv(landing_dir))
#
# print("Is streaming:", df_stream.isStreaming)
# df_stream.printSchema()

df_stream = (spark.readStream
    .schema(schema)
    .option("header", "true")           # Le fichier CSV contient une ligne d'en-tête
    .option("maxFilesPerTrigger", 1)    # Traite un seul fichier à la fois pour simuler le flux continu
    .csv(landing_dir))

print("Le dataframe est-il en streaming ? :", df_stream.isStreaming)
df_stream.printSchema()

Le dataframe est-il en streaming ? : True
root
 |-- date: timestamp (nullable = true)
 |-- team_1: string (nullable = true)
 |-- team_2: string (nullable = true)
 |-- _map: string (nullable = true)
 |-- result_1: integer (nullable = true)
 |-- result_2: integer (nullable = true)
 |-- map_winner: integer (nullable = true)
 |-- starting_ct: integer (nullable = true)
 |-- ct_1: integer (nullable = true)
 |-- t_2: integer (nullable = true)
 |-- t_1: integer (nullable = true)
 |-- ct_2: integer (nullable = true)
 |-- event_id: integer (nullable = true)
 |-- match_id: integer (nullable = true)
 |-- rank_1: integer (nullable = true)
 |-- rank_2: integer (nullable = true)
 |-- map_wins_1: integer (nullable = true)
 |-- map_wins_2: integer (nullable = true)
 |-- match_winner: integer (nullable = true)



## 3. Watermark + Windowed Aggregation

Apply a watermark on the event-time column, then group by a tumbling/sliding window and your track-specific key.

In [13]:
# TODO: Watermark + windowed aggregation
# windowed = (df_stream
#     .withWatermark("event_time", "10 minutes")
#     .groupBy(
#         F.window("event_time", "5 minutes"),  # tumbling window
#         F.col("key_col")
#     )
#     .agg(
#         F.count("*").alias("event_count"),
#         F.avg("metric").alias("avg_metric")
#     ))
#
# print("Windowed schema:")
# windowed.printSchema()

windowed = (df_stream
    # Watermark : On dit à Spark d'accepter les données en retard jusqu'à 2 jours
    .withWatermark("date", "2 days") 
    
    # GroupBy : On groupe par fenêtres temporelles de 1 jour, ET par carte jouée
    .groupBy(
        F.window("date", "1 day"),  # Tumbling window (fenêtre fixe d'un jour)
        F.col("_map")
    )
    
    # Agrégations : On compte le nombre de matchs et on fait la moyenne du score
    .agg(
        F.count("*").alias("match_count"),
        F.avg("result_1").alias("avg_score_team1")
    ))

print("Windowed schema:")
windowed.printSchema()

Windowed schema:
root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- _map: string (nullable = true)
 |-- match_count: long (nullable = false)
 |-- avg_score_team1: double (nullable = true)



## 4. Write to Parquet Sink

Write the windowed results to a Parquet sink with a trigger interval. Use `checkpointLocation` for fault tolerance.

In [14]:
# TODO: Write to Parquet sink
# pathlib.Path("outputs/lab1/stream_sink").mkdir(parents=True, exist_ok=True)
# pathlib.Path("outputs/lab1/checkpoint").mkdir(parents=True, exist_ok=True)
#
# query = (windowed.writeStream
#     .format("parquet")
#     .outputMode("append")
#     .option("path", "outputs/lab1/stream_sink")
#     .option("checkpointLocation", "outputs/lab1/checkpoint")
#     .trigger(processingTime="10 seconds")
#     .start())
#
# print("Query name:", query.name)
# print("Query id:", query.id)

# Let stream run for a fixed time, then stop
# query.awaitTermination(timeout=60)
# query.stop()

import threading
import time
import pandas as pd

# 1. On définit une petite fonction qui va appeler ton simulateur
def run_simulation():
    print("      [Simulateur] Démarrage de l'injection des fichiers...")
    # On attend 2 petites secondes pour laisser à Spark le temps de bien s'allumer
    time.sleep(2) 
    simulate_csv_streaming("data/TrackA_CSGO/results.csv", landing_dir, num_chunks=5)

# 2. On crée un "Thread" (un processus parallèle) pour la simulation
sim_thread = threading.Thread(target=run_simulation)

# 3. CONFIGURATION DU STREAM
output_path = "outputs/lab1/stream_sink"
checkpoint_path = "outputs/lab1/checkpoint"
pathlib.Path(output_path).mkdir(parents=True, exist_ok=True)
pathlib.Path(checkpoint_path).mkdir(parents=True, exist_ok=True)

# 4. LANCEMENT
print("Lancement combiné : Simulation + Streaming...")
sim_thread.start() # Lance l'injection des fichiers en arrière-plan

query = (windowed.writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", output_path)
    .option("checkpointLocation", checkpoint_path)
    .trigger(processingTime="10 seconds")
    .start())

print(f"Stream actif (ID: {query.id})")

# 5. ATTENTE ET ARRÊT
# On laisse tourner 60s pour que le thread ait le temps de finir et Spark de traiter
query.awaitTermination(timeout=60)
query.stop()

# On s'assure que le thread de simulation est bien fini
sim_thread.join()

print("Tout est terminé : fichiers injectés et stream arrêté proprement.")

Lancement combiné : Simulation + Streaming...
      [Simulateur] Démarrage de l'injection des fichiers...
Stream actif (ID: 49baa6b2-b782-4605-a151-f8f06e4e3415)


26/04/27 15:24:49 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/27 15:24:50 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


Fichier créé : data/landing/csgo_matches//match_data_part_1.csv (9154 lignes)
Fichier créé : data/landing/csgo_matches//match_data_part_2.csv (9154 lignes)
Fichier créé : data/landing/csgo_matches//match_data_part_3.csv (9154 lignes)
Fichier créé : data/landing/csgo_matches//match_data_part_4.csv (9154 lignes)
Fichier créé : data/landing/csgo_matches//match_data_part_5.csv (9157 lignes)


26/04/27 15:24:52 WARN HDFSBackedStateStoreProvider StateStoreProviderId[ storeId=StateStoreId[ operatorId=0, partitionId=0, storeName=default ], queryRunId=c97f79d2-9fc7-48ea-83e6-5d5b254e5d1c ]: The state for version 2 doesn't exist in loadedMaps. Reading snapshot file and delta files if needed...Note that this is normal for the first batch of starting query.
26/04/27 15:24:52 WARN ChecksumCheckpointFileManager: No checksum file found for file:/mnt/c/Users/flori/OneDrive/Documents/Nouvelle_Sauvegarde/ESIEE/E4FD/Data_engineering_TAJINI/DE2/outputs/lab1/checkpoint/state/0/0/2.snapshot, hence no checksum verification.
26/04/27 15:24:52 WARN ChecksumCheckpointFileManager: No checksum file found for file:/mnt/c/Users/flori/OneDrive/Documents/Nouvelle_Sauvegarde/ESIEE/E4FD/Data_engineering_TAJINI/DE2/outputs/lab1/checkpoint/state/0/0/1.snapshot, hence no checksum verification.
26/04/27 15:24:52 WARN HDFSBackedStateStoreProvider StateStoreProviderId[ storeId=StateStoreId[ operatorId=0, part

Tout est terminé : fichiers injectés et stream arrêté proprement.


26/04/27 15:25:49 WARN DAGScheduler: Failed to cancel job group c97f79d2-9fc7-48ea-83e6-5d5b254e5d1c. Cannot find active jobs for it.
26/04/27 15:25:49 WARN DAGScheduler: Failed to cancel job group c97f79d2-9fc7-48ea-83e6-5d5b254e5d1c. Cannot find active jobs for it.


## 5. Monitor — `query.lastProgress` & Streaming UI

Capture streaming metrics from `query.lastProgress` and take screenshots of the Streaming tab in Spark UI.

In [15]:
# TODO: Capture query.lastProgress
# progress = query.lastProgress
# print(json.dumps(progress, indent=2))
#
# # Save progress JSON as evidence
# pathlib.Path("proof").mkdir(parents=True, exist_ok=True)
# with open("proof/query_progress.json", "w") as f:
#     json.dump(progress, f, indent=2)
#
# # Key metrics to record:
# # - inputRowsPerSecond
# # - processedRowsPerSecond
# # - durationMs (triggerExecution, getBatch, etc.)
# # - stateOperators (numRowsTotal, numRowsUpdated)
#
# # TODO: Save streaming plan
# # windowed.explain("formatted")
# # Copy to proof/plan_streaming.txt
#
# # TODO: Open Spark UI → Streaming tab, take screenshots → proof/

import sys

# Création du dossier "proof"
proof_dir = "proof"
pathlib.Path(proof_dir).mkdir(parents=True, exist_ok=True)
print(f"Dossier de preuves '{proof_dir}/' prêt.\n")

# Récupération du dernier progrès disponible
progress = query.lastProgress 
if not progress and len(query.recentProgress) > 0:
    progress = query.recentProgress[-1]

if progress:
    print("MÉTRIQUES CLÉS DU DERNIER MICRO-BATCH :")
    print(f"  - Vitesse d'entrée (inputRowsPerSecond) : {progress.get('inputRowsPerSecond')}")
    print(f"  - Vitesse de traitement (processedRowsPerSecond) : {progress.get('processedRowsPerSecond')}")
    
    if 'durationMs' in progress:
        print(f"  - Durée du trigger (triggerExecution) : {progress['durationMs'].get('triggerExecution')} ms")
        print(f"  - Durée de lecture du batch (getBatch) : {progress['durationMs'].get('getBatch')} ms")
        
    if 'stateOperators' in progress and len(progress['stateOperators']) > 0:
        state_op = progress['stateOperators'][0]
        print(f"  - Lignes maintenues en mémoire (numRowsTotal) : {state_op.get('numRowsTotal')}")
        print(f"  - Lignes mises à jour (numRowsUpdated) : {state_op.get('numRowsUpdated')}")
        
    # Sauvegarde dans le fichier JSON
    json_path = f"{proof_dir}/query_progress.json"
    with open(json_path, "w") as f:
        json.dump(progress, f, indent=2, default=str)
    print(f"\nPreuve sauvegardée : {json_path}")
else:
    print("Aucun log trouvé.")

plan_path = f"{proof_dir}/plan_streaming.txt"

# On redirige silencieusement l'affichage du plan dans le fichier texte
with open(plan_path, "w") as f:
    original_stdout = sys.stdout
    sys.stdout = f
    windowed.explain("formatted")
    sys.stdout = original_stdout

print(f"Preuve sauvegardée : {plan_path}")

Dossier de preuves 'proof/' prêt.

MÉTRIQUES CLÉS DU DERNIER MICRO-BATCH :
  - Vitesse d'entrée (inputRowsPerSecond) : 494.1
  - Vitesse de traitement (processedRowsPerSecond) : 517.1
  - Durée du trigger (triggerExecution) : 17708 ms
  - Durée de lecture du batch (getBatch) : 216 ms
  - Lignes maintenues en mémoire (numRowsTotal) : 17
  - Lignes mises à jour (numRowsUpdated) : 0

Preuve sauvegardée : proof/query_progress.json
Preuve sauvegardée : proof/plan_streaming.txt


## 6. Before/After Optimization

Re-run the pipeline with a different trigger interval or watermark duration. Compare throughput and latency metrics.

**Before:** `trigger(processingTime="10 seconds")`, watermark `"10 minutes"`
**After:** Change one parameter (e.g., `trigger(processingTime="30 seconds")` or watermark `"5 minutes"`) and compare.

In [ ]:
# TODO: Re-run with different trigger/watermark, capture metrics
# Compare inputRowsPerSecond, processedRowsPerSecond, durationMs
# Record both runs in lab1_metrics_log.csv

# Example metrics log structure:
# run_id, trigger_interval, watermark_duration, inputRowsPerSecond,
# processedRowsPerSecond, numInputRows, stateRows, durationMs, timestamp

import csv
from datetime import datetime

# Paramètres de notre Run Optimisé (Run 2)
RUN_ID = "run_2_optimized"
TRIGGER_INTERVAL = "5 seconds"   # Plus rapide que les 10s précédentes
WATERMARK_DURATION = "1 hour"    # Tolérance réduite par rapport aux 2 jours

# Nouveaux dossiers pour éviter les conflits avec le run précédent
output_path_run2 = "outputs/lab1/stream_sink_run2"
checkpoint_path_run2 = "outputs/lab1/checkpoint_run2"

print(f"Lancement du {RUN_ID}")

# Redéfinition du flux avec les nouveaux paramètres
windowed_run2 = (df_stream
    .withWatermark("date", WATERMARK_DURATION) 
    .groupBy(F.window("date", "1 day"), F.col("_map"))
    .agg(F.count("*").alias("match_count"), F.avg("result_1").alias("avg_score_team1")))

# Démarrage du flux
query_run2 = (windowed_run2.writeStream
    .format("parquet")
    .outputMode("append") 
    .option("path", output_path_run2)
    .option("checkpointLocation", checkpoint_path_run2)
    .trigger(processingTime=TRIGGER_INTERVAL)
    .start())

print("Le stream tourne pendant 30 secondes pour capter les métriques")
time.sleep(30)
query_run2.stop()
print("Stream arrêté.")

progress = query_run2.lastProgress or (query_run2.recentProgress[-1] if query_run2.recentProgress else None)

if progress:
    # Extraction sécurisée des données
    input_rate = progress.get('inputRowsPerSecond', 0)
    process_rate = progress.get('processedRowsPerSecond', 0)
    num_input_rows = progress.get('numInputRows', 0)
    duration_ms = progress.get('durationMs', {}).get('triggerExecution', 0)
    
    state_rows = 0
    if 'stateOperators' in progress and len(progress['stateOperators']) > 0:
        state_rows = progress['stateOperators'][0].get('numRowsTotal', 0)
        
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Écriture dans le CSV
    csv_file = "lab1_metrics_log.csv"
    file_exists = os.path.isfile(csv_file)
    
    with open(csv_file, mode='a', newline='') as f:
        writer = csv.writer(f)
        # Écriture de l'en-tête si le fichier est nouveau
        if not file_exists:
            writer.writerow(["run_id", "trigger_interval", "watermark_duration", "inputRowsPerSecond", 
                             "processedRowsPerSecond", "numInputRows", "stateRows", "durationMs", "timestamp"])
        
        # Écriture de la ligne de log
        writer.writerow([RUN_ID, TRIGGER_INTERVAL, WATERMARK_DURATION, input_rate, 
                         process_rate, num_input_rows, state_rows, duration_ms, "Optimized run", timestamp])
        
    print(f"Métriques enregistrées avec succès dans {csv_file}")
else:
    print("Aucune métrique trouvée (aucun fichier CSV n'a été traité pendant les 30s).")

spark.stop()
print("Lab 1 complete.")

Lancement du run_2_optimized
Le stream tourne pendant 30 secondes pour capter les métriques


26/04/27 15:25:57 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/27 15:25:58 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.
26/04/27 15:25:59 WARN HDFSBackedStateStoreProvider StateStoreProviderId[ storeId=StateStoreId[ operatorId=0, partitionId=10, storeName=default ], queryRunId=2d13abb8-dda3-4707-80c6-01116c5b1431 ]: The state for version 1 doesn't exist in loadedMaps. Reading snapshot file and delta files if needed...Note that this is normal for the first batch of starting query.
26/04/27 15:25:59 WARN ChecksumCheckpointFileManager: No checksum file found for file:/mnt/c/Users/flori/OneDrive/Documents/Nouvelle_Sauvegarde/ESIEE/E4FD/Data_engineering_TAJINI/DE2/outputs/lab1/checkpoint_run2/state/0/10/1.snapshot, hence no checksum verification.
26/04/27 15:25:59 WARN HDFSBackedStateStoreProvider StateStoreProviderId[ storeId=StateStoreId[ operatorId=0, partit

Stream arrêté.
Métriques enregistrées avec succès dans lab1_metrics_log.csv
Lab 1 complete.


26/04/27 15:26:27 WARN DAGScheduler: Failed to cancel job group 2d13abb8-dda3-4707-80c6-01116c5b1431. Cannot find active jobs for it.
26/04/27 15:26:27 WARN TaskSetManager: Lost task 28.0 in stage 9.0 (TID 833) (localhost executor driver): TaskKilled (Stage cancelled: [SPARK_JOB_CANCELLED] Job 4 cancelled Query [id = 879103d9-b404-4c36-865e-5df6bb30d507, runId = 2d13abb8-dda3-4707-80c6-01116c5b1431] was stopped SQLSTATE: XXKDA)
26/04/27 15:26:27 ERROR TaskSchedulerImpl: Exception in statusUpdate+ 10) / 200]
java.util.concurrent.RejectedExecutionException: Task org.apache.spark.scheduler.TaskResultGetter$$Lambda$5227/0x0000729e7547f8a8@4175abeb rejected from java.util.concurrent.ThreadPoolExecutor@2f18099f[Terminated, pool size = 0, active threads = 0, queued tasks = 0, completed tasks = 830]
	at java.base/java.util.concurrent.ThreadPoolExecutor$AbortPolicy.rejectedExecution(ThreadPoolExecutor.java:2065)
	at java.base/java.util.concurrent.ThreadPoolExecutor.reject(ThreadPoolExecutor.jav

26/04/27 15:26:51 WARN StateStore: Error running maintenance thread
java.lang.IllegalStateException: SparkEnv not active, cannot do maintenance on StateStores
	at org.apache.spark.sql.execution.streaming.state.StateStore$.doMaintenance(StateStore.scala:1440)
	at org.apache.spark.sql.execution.streaming.state.StateStore$.$anonfun$startMaintenanceIfNeeded$1(StateStore.scala:1386)
	at org.apache.spark.sql.execution.streaming.state.StateStore$MaintenanceTask$$anon$1.run(StateStore.scala:1079)
	at java.base/java.util.concurrent.Executors$RunnableAdapter.call(Executors.java:539)
	at java.base/java.util.concurrent.FutureTask.runAndReset(FutureTask.java:305)
	at java.base/java.util.concurrent.ScheduledThreadPoolExecutor$ScheduledFutureTask.run(ScheduledThreadPoolExecutor.java:305)
	at java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.java:1136)
	at java.base/java.util.concurrent.ThreadPoolExecutor$Worker.run(ThreadPoolExecutor.java:635)
	at java.base/java.lang.Thre